In [5]:
# Ensure notebook runs from repository root and imports resolve
import os
import sys

def find_repo_root(start=None):
    start = start or os.getcwd()
    cur = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(cur, 'pyproject.toml')) or os.path.exists(os.path.join(cur, 'README.md')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            return os.path.abspath(start)
        cur = parent

repo_root = find_repo_root()
print('Setting repo root to', repo_root)
if os.getcwd() != repo_root:
    os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)


Setting repo root to /home/lgreen/projects/Online_BS


# Train LeNet5 on MNIST
Train for 100 epochs using `AdamW` (default params) and batch size 32.
Saves the trained model to /home/lgreen/projects/Online_BS/models/teacher/MNIST.tar and prints final training and validation accuracies.

In [6]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from models.LeNet import create_model
from tqdm import tqdm

In [7]:
# Data transforms and loaders
batch_size = 32
epochs = 10
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # LeNet expects 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
data_root = os.path.join(os.getcwd(), 'data')
train_dataset = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
val_dataset = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

In [8]:
# Build model, optimizer, loss
# Enforce use of CUDA device 1
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available; training requires cuda:1')
ngpu = torch.cuda.device_count()
if ngpu <= 1:
    raise RuntimeError(f'CUDA device 1 not available; found {ngpu} GPU(s)')

device = torch.device('cuda:1')
print('Using device', device)

model = create_model(in_channels=1, num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()

# Training loop with tqdm progress bars
for epoch in range(1, epochs + 1):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())

# After training: evaluate on train and validation sets
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            out = model(xb)
            preds = out.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

train_acc = evaluate(train_loader)
val_acc = evaluate(val_loader)

# Save model state_dict (plain mapping) so it can be loaded directly with torch.load(path)
save_path = os.path.join(repo_root, 'models', 'teacher', 'MNIST.tar') if 'repo_root' in globals() else '/home/lgreen/projects/Online_BS/models/teacher/MNIST.tar'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)

print(f'Final training accuracy: {train_acc * 100:.2f}%')
print(f'Final validation accuracy: {val_acc * 100:.2f}%')
print(f'Saved model to: {save_path}')


Using device cuda:1


Final training accuracy: 99.34%
Final validation accuracy: 98.92%
Saved model to: /home/lgreen/projects/Online_BS/models/teacher/MNIST.tar
